### 保有銘柄集計機能
＜仕様＞  
・銘柄コードから、IDmapに登録されたURLを取得し、該当ページのIRBANKのページにアクセス  
・必要な情報を取得(今回は、配当金額を取得し、予定と実績を分けて取得)  

＜追加機能＞
・将来的には、他の指標も取得。

In [17]:
# ==========================================
# 配当取得 & 03_haitou.csv（Excel見やすい横持ち）更新
# 追加：
# - 連続アクセス対策 sleep（ジッター）
# - 失敗時リトライ（指数バックオフ+ジッター）
# - 1銘柄ごとにCSV保存
# - 既取得スキップ
# - ログ/集計を必ず出す
# ==========================================

import os
import csv
import re
import time
import random
from dataclasses import dataclass
from datetime import datetime
from typing import Dict, List, Optional, Tuple

import requests
from bs4 import BeautifulSoup


# (1) 設定
IDMAP_CSV = os.path.join("Data", "01_IDmap.csv")
HAITOU_CSV = os.path.join("Data", "03_haitou.csv")

REQUEST_TIMEOUT_SEC = 20
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"

# アクセス制御
SLEEP_SEC_BASE = 0.8
SLEEP_SEC_JITTER = 0.6
RETRY_MAX = 3
RETRY_BACKOFF_BASE = 1.2
RETRY_BACKOFF_FACTOR = 2.0


# (2) データ構造
@dataclass
class IdMapRow:
    code: int
    name: str
    url: str
    status: str


# (3) ユーティリティ
def _safe_int(s: str) -> Optional[int]:
    try:
        return int(s)
    except Exception:
        return None


def _normalize_space(s: str) -> str:
    s = s.replace("\u00a0", " ")
    s = re.sub(r"[ \t\r\n]+", " ", s)
    return s.strip()


def _parse_year_month_from_dt(dt_text: str) -> Optional[Tuple[int, int]]:
    t = _normalize_space(dt_text)

    m = re.search(r"(\d{4})/(\d{2})", t)
    if m:
        return int(m.group(1)), int(m.group(2))

    m2 = re.search(r"(\d{2})/(\d{2})", t)
    if m2:
        yy = int(m2.group(1))
        mm = int(m2.group(2))
        return 2000 + yy, mm

    return None


def _extract_amount_from_dd(dd_text: str) -> Optional[str]:
    t = _normalize_space(dd_text)
    t = t.replace("円", "")
    t = t.replace("*", "")
    m = re.search(r"(\d+(?:\.\d+)?)", t)
    return m.group(1) if m else None


def _is_valid_irbank_url(url: str) -> bool:
    u = (url or "").strip()
    if not u or u == "0":
        return False
    if "irbank.net/" not in u:
        return False
    return True


def _sleep_polite() -> None:
    sec = SLEEP_SEC_BASE + random.random() * SLEEP_SEC_JITTER
    time.sleep(sec)


def _backoff_sleep(attempt_index: int) -> None:
    sec = (RETRY_BACKOFF_BASE * (RETRY_BACKOFF_FACTOR ** attempt_index)) + (random.random() * SLEEP_SEC_JITTER)
    time.sleep(sec)


# (4) IDmap
def load_idmap(idmap_csv: str = IDMAP_CSV) -> Dict[int, IdMapRow]:
    if not os.path.exists(idmap_csv):
        raise FileNotFoundError(f"IDmapが見つかりません: {idmap_csv}")

    with open(idmap_csv, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        required = {"code", "URL", "name", "status"}
        for col in required:
            if col not in (reader.fieldnames or []):
                raise ValueError(f"IDmapに必要な列がありません: {col} / columns={reader.fieldnames}")

        out: Dict[int, IdMapRow] = {}
        for row in reader:
            code = _safe_int((row.get("code") or "").strip())
            if code is None:
                continue
            out[code] = IdMapRow(
                code=code,
                name=(row.get("name") or "").strip(),
                url=(row.get("URL") or "").strip(),
                status=(row.get("status") or "").strip(),
            )
        return out


def select_target_codes_from_idmap(idmap: Dict[int, IdMapRow]) -> List[int]:
    targets: List[int] = []
    for code, info in idmap.items():
        if info.status.strip().lower() == "skip":
            continue
        if not _is_valid_irbank_url(info.url):
            continue
        targets.append(code)
    return sorted(set(targets))


# (5) 取得（リトライ付き）
def fetch_irbank_html(url: str) -> str:
    headers = {
        "User-Agent": USER_AGENT,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    }

    last_err: Optional[Exception] = None

    for attempt in range(RETRY_MAX):
        try:
            _sleep_polite()
            r = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT_SEC)
            r.raise_for_status()
            return r.text
        except Exception as e:
            last_err = e
            if attempt >= RETRY_MAX - 1:
                break
            _backoff_sleep(attempt_index=attempt)

    raise RuntimeError(f"fetch failed after {RETRY_MAX} attempts: {url} / last_err={last_err}")


# (6) DOMで「一株配当」を抽出
def find_dividend_dl(soup: BeautifulSoup):
    """
    「一株配当」の dl.gdl を返す。
    ★id(c_23) は銘柄によって別項目になるので信用しない
    """
    # まず h2 の文言で確実に特定する
    h2 = soup.find(lambda tag: tag.name == "h2" and "一株配当" in tag.get_text(" ", strip=True))
    if h2:
        container = h2.find_parent("div")
        if container:
            dl = container.find("dl", class_="gdl")
            if dl:
                return dl

    # どうしても見つからない場合のみ、idで探す（ただし後段で「円」ガードが効く）
    div = soup.find("div", id="c_23")
    if div:
        dl = div.find("dl", class_="gdl")
        if dl:
            return dl

    return None

def parse_dividends_yearly_from_irbank_html(html: str) -> Dict[int, str]:
    soup = BeautifulSoup(html, "html.parser")
    dl = find_dividend_dl(soup)
    if dl is None:
        return {}

    items: List[Tuple[int, int, bool, str]] = []
    current_dt_text: Optional[str] = None

    for child in dl.find_all(["dt", "dd"], recursive=False):
        if child.name == "dt":
            current_dt_text = child.get_text(" ", strip=True)
            continue

        if child.name == "dd" and current_dt_text:
            ym = _parse_year_month_from_dt(current_dt_text)
            if ym is None:
                current_dt_text = None
                continue

            year, month = ym
            is_forecast = ("予" in current_dt_text)

            # ★ここが重要：「円」が無い dd は配当ではないので捨てる（%系の誤検知排除）
            dd_raw = child.get_text(" ", strip=True)
            if "円" not in dd_raw:
                current_dt_text = None
                continue

            amount = _extract_amount_from_dd(dd_raw)
            current_dt_text = None

            if amount is None:
                continue

            items.append((year, month, is_forecast, amount))

    if not items:
        return {}

    best: Dict[int, Tuple[int, bool, str]] = {}
    for year, month, is_forecast, amount_str in items:
        if year not in best:
            best[year] = (month, is_forecast, amount_str)
            continue

        cur_month, cur_is_forecast, _ = best[year]

        if cur_is_forecast and (not is_forecast):
            best[year] = (month, is_forecast, amount_str)
            continue
        if (not cur_is_forecast) and is_forecast:
            continue
        if month >= cur_month:
            best[year] = (month, is_forecast, amount_str)

    out: Dict[int, str] = {}
    for year, (_, is_forecast, amount_str) in best.items():
        out[year] = f"予 {amount_str}" if is_forecast else amount_str
    return out

# (7) CSV I/O
def load_haitou_wide(haitou_csv: str = HAITOU_CSV) -> Tuple[List[str], Dict[int, Dict[str, str]]]:
    if not os.path.exists(haitou_csv):
        return ["code", "name"], {}

    with open(haitou_csv, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        header = reader.fieldnames or ["code", "name"]
        data: Dict[int, Dict[str, str]] = {}
        for row in reader:
            code = _safe_int((row.get("code") or "").strip())
            if code is None:
                continue
            data[code] = {k: (row.get(k) or "").strip() for k in header}
        return header, data


def save_haitou_wide(header: List[str], data: Dict[int, Dict[str, str]], haitou_csv: str = HAITOU_CSV) -> None:
    os.makedirs(os.path.dirname(haitou_csv), exist_ok=True)
    with open(haitou_csv, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=header)
        writer.writeheader()
        for code in sorted(data.keys()):
            row = data[code]
            writer.writerow({col: row.get(col, "") for col in header})


def normalize_year_columns(header: List[str], data: Dict[int, Dict[str, str]], keep_past_years: int = 10) -> List[str]:
    fixed_cols = ["code", "name"]

    years: List[int] = []
    for col in header:
        if col in fixed_cols:
            continue
        y = _safe_int(col)
        if y is not None:
            years.append(y)

    if not years:
        return fixed_cols

    max_year = max(years)
    min_keep = max_year - (keep_past_years - 1)

    kept_years = sorted([y for y in set(years) if (y >= min_keep)])
    new_header = fixed_cols + [str(y) for y in kept_years]

    keep_set = set(new_header)
    for code, row in data.items():
        for k in list(row.keys()):
            if k not in keep_set:
                row.pop(k, None)
        row["code"] = str(code)
        row["name"] = row.get("name", "")

    return new_header


def merge_cell(existing: str, incoming: str) -> str:
    existing = (existing or "").strip()
    incoming = (incoming or "").strip()
    if incoming == "":
        return existing

    incoming_is_num = re.fullmatch(r"\d+(?:\.\d+)?", incoming) is not None
    existing_is_forecast = existing.startswith("予 ")

    if existing_is_forecast and incoming_is_num:
        return incoming

    return incoming


# (8) ★主要処理（ログ確実・1銘柄ごと保存・既取得スキップ）
def update_haitou_for_codes(codes: List[int], idmap_csv: str = IDMAP_CSV, haitou_csv: str = HAITOU_CSV) -> None:
    idmap = load_idmap(idmap_csv)

    cnt_ok = 0
    cnt_skip_got = 0
    cnt_skip_nourl = 0
    cnt_skip_nodiv = 0
    cnt_skip_other = 0
    cnt_err = 0

    print(f"[START] update_haitou_for_codes: targets={len(codes)}")

    for idx, code in enumerate(codes, start=1):
        header, data = load_haitou_wide(haitou_csv)

        info = idmap.get(code)
        if info is None:
            cnt_skip_other += 1
            print(f"[SKIP] ({idx}/{len(codes)}) IDmapなし code={code}")
            continue

        if info.status.strip().lower() == "skip":
            cnt_skip_other += 1
            print(f"[SKIP] ({idx}/{len(codes)}) status=skip code={code} name={info.name}")
            continue

        if not _is_valid_irbank_url(info.url):
            cnt_skip_nourl += 1
            print(f"[SKIP] ({idx}/{len(codes)}) URL無効 code={code} name={info.name}")
            continue

        # 既取得判定：年列のどれか1つでも埋まっていれば取得済とみなす（安全）
        if code in data:
            year_cols = [c for c in header if c not in ("code", "name") and _safe_int(c) is not None]
            any_filled = any(((data[code].get(y) or "").strip() != "") for y in year_cols)
            if any_filled:
                cnt_skip_got += 1
                print(f"[SKIP] ({idx}/{len(codes)}) 取得済 code={code} name={info.name}")
                continue

        try:
            html = fetch_irbank_html(info.url)
            year_to_cell = parse_dividends_yearly_from_irbank_html(html)
        except Exception as e:
            cnt_err += 1
            print(f"[ERROR] ({idx}/{len(codes)}) 取得失敗 code={code} name={info.name} err={e}")
            continue

        if not year_to_cell:
            cnt_skip_nodiv += 1
            print(f"[SKIP] ({idx}/{len(codes)}) 無配/配当情報なし code={code} name={info.name}")
            continue

        # 反映
        if code not in data:
            data[code] = {"code": str(code), "name": info.name}
        else:
            data[code]["code"] = str(code)
            data[code]["name"] = info.name

        existing_years = set(_safe_int(c) for c in header if _safe_int(c) is not None)

        for y in sorted(set(year_to_cell.keys()) - set(existing_years)):
            header.append(str(y))

        for y, cell in year_to_cell.items():
            col = str(y)
            data[code][col] = merge_cell(data[code].get(col, ""), cell)

        # 保存
        fixed = ["code", "name"]
        years_sorted = sorted([_safe_int(c) for c in header if c not in fixed and _safe_int(c) is not None])
        header = fixed + [str(y) for y in years_sorted]

        header = normalize_year_columns(header, data, keep_past_years=10)
        save_haitou_wide(header, data, haitou_csv)

        cnt_ok += 1
        print(f"[OK] ({idx}/{len(codes)}) 保存完了 code={code} name={info.name} years={min(year_to_cell)}..{max(year_to_cell)}")

    print(
        "[DONE] "
        f"ok={cnt_ok} skip_got={cnt_skip_got} skip_nourl={cnt_skip_nourl} "
        f"skip_nodiv={cnt_skip_nodiv} skip_other={cnt_skip_other} err={cnt_err}"
    )


# (9) 実行
if __name__ == "__main__":
    idmap_all = load_idmap(IDMAP_CSV)
    target_codes = select_target_codes_from_idmap(idmap_all)
    print(f"[INFO] 対象銘柄数（URL有効）: {len(target_codes)}")
    update_haitou_for_codes(target_codes, IDMAP_CSV, HAITOU_CSV)


[INFO] 対象銘柄数（URL有効）: 3778
[START] update_haitou_for_codes: targets=3778
[SKIP] (1/3778) 無配/配当情報なし code=130 name=Veritas In Silico
[SKIP] (2/3778) 無配/配当情報なし code=135 name=VRAIN Solution
[SKIP] (3/3778) 無配/配当情報なし code=137 name=Cocolive
[SKIP] (4/3778) 取得済 code=138 name=光フードサービス
[SKIP] (5/3778) 取得済 code=141 name=トライアル HD
[SKIP] (6/3778) 無配/配当情報なし code=142 name=ジンジブ
[SKIP] (7/3778) 無配/配当情報なし code=143 name=イシン
[SKIP] (8/3778) 無配/配当情報なし code=145 name=L is B
[SKIP] (9/3778) 取得済 code=146 name=コロンビア・ワークス
[SKIP] (10/3778) 無配/配当情報なし code=147 name=ソラコム
[SKIP] (11/3778) 無配/配当情報なし code=148 name=ハッチ・ワーク
[SKIP] (12/3778) 無配/配当情報なし code=149 name=シンカ
[SKIP] (13/3778) 無配/配当情報なし code=150 name=JSH
[SKIP] (14/3778) 無配/配当情報なし code=151 name=ダイブ
[SKIP] (15/3778) 取得済 code=153 name=カウリス


KeyboardInterrupt: 

### 全銘柄を再取得（試していない）

In [ ]:
def update_haitou_for_codes(
    codes: List[int],
    idmap_csv: str = IDMAP_CSV,
    haitou_csv: str = HAITOU_CSV,
    force_refresh: bool = False,   # ★追加：Trueなら既取得でも再取得
) -> None:
    idmap = load_idmap(idmap_csv)

    cnt_ok = 0
    cnt_skip_got = 0
    cnt_skip_nourl = 0
    cnt_skip_nodiv = 0
    cnt_skip_other = 0
    cnt_err = 0

    print(f"[START] update_haitou_for_codes: targets={len(codes)} force_refresh={force_refresh}")

    for idx, code in enumerate(codes, start=1):
        header, data = load_haitou_wide(haitou_csv)

        info = idmap.get(code)
        if info is None:
            cnt_skip_other += 1
            print(f"[SKIP] ({idx}/{len(codes)}) IDmapなし code={code}")
            continue

        if info.status.strip().lower() == "skip":
            cnt_skip_other += 1
            print(f"[SKIP] ({idx}/{len(codes)}) status=skip code={code} name={info.name}")
            continue

        if not _is_valid_irbank_url(info.url):
            cnt_skip_nourl += 1
            print(f"[SKIP] ({idx}/{len(codes)}) URL無効 code={code} name={info.name}")
            continue

        # ★既取得スキップ（force_refresh=False のときだけ）
        if (not force_refresh) and (code in data):
            year_cols = [c for c in header if c not in ("code", "name") and _safe_int(c) is not None]
            any_filled = any(((data[code].get(y) or "").strip() != "") for y in year_cols)
            if any_filled:
                cnt_skip_got += 1
                print(f"[SKIP] ({idx}/{len(codes)}) 取得済 code={code} name={info.name}")
                continue

        try:
            html = fetch_irbank_html(info.url)
            year_to_cell = parse_dividends_yearly_from_irbank_html(html)
        except Exception as e:
            cnt_err += 1
            print(f"[ERROR] ({idx}/{len(codes)}) 取得失敗 code={code} name={info.name} err={e}")
            continue

        if not year_to_cell:
            cnt_skip_nodiv += 1
            print(f"[SKIP] ({idx}/{len(codes)}) 無配/配当情報なし code={code} name={info.name}")
            continue

        if code not in data:
            data[code] = {"code": str(code), "name": info.name}
        else:
            data[code]["code"] = str(code)
            data[code]["name"] = info.name

        existing_years = set(_safe_int(c) for c in header if _safe_int(c) is not None)

        for y in sorted(set(year_to_cell.keys()) - set(existing_years)):
            header.append(str(y))

        for y, cell in year_to_cell.items():
            col = str(y)
            data[code][col] = merge_cell(data[code].get(col, ""), cell)

        fixed = ["code", "name"]
        years_sorted = sorted([_safe_int(c) for c in header if c not in fixed and _safe_int(c) is not None])
        header = fixed + [str(y) for y in years_sorted]

        header = normalize_year_columns(header, data, keep_past_years=10)
        save_haitou_wide(header, data, haitou_csv)

        cnt_ok += 1
        print(f"[OK] ({idx}/{len(codes)}) 保存完了 code={code} name={info.name} years={min(year_to_cell)}..{max(year_to_cell)}")

    print(
        "[DONE] "
        f"ok={cnt_ok} skip_got={cnt_skip_got} skip_nourl={cnt_skip_nourl} "
        f"skip_nodiv={cnt_skip_nodiv} skip_other={cnt_skip_other} err={cnt_err}"
    )

if __name__ == "__main__":
    # ==========
    # 将来の総点検（例：1年後に全銘柄をなめて更新）
    # ==========
    FORCE_REFRESH_ALL = True   # ★ここを True にすると「取得済スキップしない」

    idmap_all = load_idmap(IDMAP_CSV)
    target_codes = select_target_codes_from_idmap(idmap_all)

    print(f"[INFO] 対象銘柄数（URL有効）: {len(target_codes)}")
    update_haitou_for_codes(
        target_codes,
        idmap_csv=IDMAP_CSV,
        haitou_csv=HAITOU_CSV,
        force_refresh=FORCE_REFRESH_ALL,
    )
